In [1]:
from train_model import *


In [7]:
n_mimo = 4
save_model_folder = os.path.join(project_root, "src", "models", "fitted_models")

csv_semi_processed_path = os.path.join(project_root, "data", "processed", "semi_processed.csv")
df = pd.read_csv(csv_semi_processed_path, encoding='utf-8')

all_df = Data_selector(df).select_peaks(goodness=3)

# train_model(train_df, n_mimo, n_est=2000, m_depth=7, save_model=save_model, save_model_folder=save_model_folder)

model = load_model(save_model_folder)

base_features = ["name", "code", "temperature", "humidity", "dew", "surface_pressure", "value",
                     "forecast", "status", "season", "datetime", "generation_with_24_delay"]
base_feature_selector = Feature_selector(all_df, target="generation")
base_feature_selector.select(features_to_select=base_features)
df_selected = base_feature_selector.df.copy()

logger.info(f"Test model: Some features have been dropped successfully")

feature_selector = Feature_selector(df_selected, target="generation")
Xs_test, ys_test, name_code_df = feature_selector.get_X_and_y(n_mimo=n_mimo)

ys_pred = model.predict(Xs_test)
y_pred = make_y_flatten(all_df, feature_selector, name_code_df, n_mimo, ys_pred)
df.loc[all_df.index, "prediction"] = y_pred


2025-09-27 16:18:19 - train_model - INFO - Test model: Some features have been dropped successfully


In [9]:
project_root = "U:/ML_project/bargh"
csv_write_path = os.path.join(project_root, "data", "processed", "data_for_plot.csv")
df.to_csv(csv_write_path, index=False)